# Acne improvement, Phase 3: the fresh untouched test set (scored once, two models)

**What it is.** About 100 photos that no model has ever seen or been tuned on: positives (acne photos) and clear-skin negatives. You boxed them from scratch (no model drafts). The notebook scores **two** models on exactly the same photos, once:
- **candidate_960**: the new 960 px 3-model average from Phase 2 (threshold 0.225, chosen on validation)
- **installed_800**: the 3-model average currently installed in the app (threshold 0.275, chosen on validation)

**Never run this before the 100 photos are boxed and saved.** It scores once and stores the result in a lock file; after that it only shows the stored result.

**The decision rule (fixed before any fresh-test score exists):** install the candidate only if on the fresh test its mAP@0.5 is at least **+0.02** above the installed model, its precision and recall are each no more than 0.02 lower, and its false-alarm rate on the clear-skin photos is no more than 5 points higher. Otherwise the installed model stays. The result is reported either way, including whether it reaches 0.50.

**Before you start (important):**
- Make sure no other training is running (`ps aux | grep -E "nbconvert|lesion_exp"` in WSL). WSL has only about 6.7 GB of memory; two trainings at once can freeze or crash both.
- Kernel: choose the WSL environment `.venv_det` (Python 3.11).
- Run the cells in order. Every finished epoch prints one line; `*` marks a new best mAP@0.5.
- Close other heavy programs (browsers with many tabs, other notebooks) while training.

In [ ]:
import os, sys, json, time, shutil
os.environ["YOLO_VERBOSE"] = "False"          # no progress bars: one clean line per finished epoch is printed instead
from pathlib import Path
sys.path[:0] = ["/mnt/c/Users/Bautista/source/repos/SkinDevApp/" + _d for _d in ("shared", "acne/scripts", "hyperpigmentation/scripts", "eczema/scripts", "skintone", "gradcam")]
HOME = Path.home() / "PrecisionSkinV1"

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import lesion_exp as LE
ALLOW_RESCORE = False
SMOKE = os.environ.get("NB_FRESH_FOLDER") is not None        # only used to check that the notebook runs end to end
FRESH_FOLDER = Path(os.environ.get("NB_FRESH_FOLDER", "/mnt/c/Users/Bautista/Downloads/PrecisionSkin/Acne_Boxing/Acne_Fresh_Test"))
FRESH_DIR = HOME / "lesion_experiments" / "data" / ("acne_fresh_test_SMOKE" if SMOKE else "acne_fresh_test")
LOCK = HOME / "runs" / ("acne_FRESHTEST_SMOKE_scored.json" if SMOKE else "acne_FRESHTEST_scored.json")
MODELS = {
    "candidate_960": dict(label="960 px candidate (Phase 2)", imgsz=960, thr=0.225, weights=[LE.RUNS / ("acne_final_960_lowlr_seed%d" % s) / "weights" / "best.pt" for s in (42, 7, 123)]),
    "installed_800": dict(label="installed baseline", imgsz=800, thr=0.275, weights=[HOME / "runs" / "lesion_acne_boxed_only" / "models" / ("seed_%d" % s) / "weights" / "best.pt" for s in (123, 42, 7)]),
}
for k, m in MODELS.items(): print(k, "| image size", m["imgsz"], "| threshold", m["thr"], "| weights found:", [p.exists() for p in m["weights"]])

## 1. Toolbox

In [ ]:
import json, csv, os, sys, time, shutil, random, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
warnings.filterwarnings("ignore")
EXT = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

def rd(p):
    p = Path(p); return [tuple(map(float, l.split()[1:5])) for l in p.read_text().splitlines() if len(l.split()) >= 5] if p.exists() else []
def xyxy(b): return b[0] - b[2] / 2, b[1] - b[3] / 2, b[0] + b[2] / 2, b[1] + b[3] / 2
def iou(a, b):
    a, b = xyxy(a), xyxy(b); iw, ih = max(0, min(a[2], b[2]) - max(a[0], b[0])), max(0, min(a[3], b[3]) - max(a[1], b[1])); i = iw * ih
    return i / ((a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - i + 1e-12)
def side(b): return (b[2] * b[3]) ** 0.5
def images_of(d): return sorted(p for p in (Path(d) / "images").glob("*") if p.suffix.lower() in EXT)

def raw_preds(weights, d, imgsz=800):
    """all boxes down to confidence 0.001 for every photo of folder d (normalised x-centre, y-centre, w, h)"""
    from ultralytics import YOLO
    import torch
    m = YOLO(str(weights)); dev = 0 if torch.cuda.is_available() else "cpu"; out = {}; ims = images_of(d)
    for s in range(0, len(ims), 8):
        ch = ims[s:s + 8]
        for p, r in zip(ch, m.predict([str(x) for x in ch], imgsz=imgsz, conf=0.001, iou=0.6, max_det=300, verbose=False, device=dev)):
            out[p.name] = [(tuple(map(float, b)), float(c)) for b, c in zip(r.boxes.xywhn.cpu().numpy(), r.boxes.conf.cpu().numpy())] if len(r.boxes) else []
    return out

def fuse(lists, thr=0.5):
    """weighted box fusion: boxes of the models that overlap (IoU >= thr) become one box; confidence = sum of confidences / number of models"""
    n = len(lists); allb = sorted([b for l in lists for b in l], key=lambda x: -x[1]); clusters = []
    for b, c in allb:
        for cl in clusters:
            if iou(cl["box"], b) >= thr:
                cl["m"].append((b, c)); w = np.array([x[1] for x in cl["m"]]); cl["box"] = tuple(np.average(np.array([x[0] for x in cl["m"]]), axis=0, weights=w)); break
        else: clusters.append({"box": b, "m": [(b, c)]})
    return [(cl["box"], min(1.0, sum(x[1] for x in cl["m"]) / n)) for cl in clusters]

def _ap(conf, tp, n_gt):
    if n_gt == 0 or len(conf) == 0: return 0.0
    o = np.argsort(-conf); tp = tp[o]; tpc = np.cumsum(tp); fpc = np.cumsum(1 - tp); rec = tpc / n_gt; prec = tpc / (tpc + fpc)
    mrec = np.concatenate([[0], rec, [1]]); mpre = np.concatenate([[1], prec, [0]]); mpre = np.flip(np.maximum.accumulate(np.flip(mpre))); x = np.linspace(0, 1, 101)
    return float(np.trapezoid(np.interp(x, mrec, mpre), x)) if hasattr(np, "trapezoid") else float(np.trapz(np.interp(x, mrec, mpre), x))

def match_flags(gt, pr, thr):
    """greedy by confidence: returns (confidences, 1/0 true-positive flags) of every predicted box of one photo"""
    used = set(); cs, fl = [], []
    for b, c in sorted(pr, key=lambda x: -x[1]):
        best, bj = 0, -1
        for j, g in enumerate(gt):
            if j in used: continue
            v = iou(g, b)
            if v > best: best, bj = v, j
        ok = bj >= 0 and best >= thr
        if ok: used.add(bj)
        cs.append(c); fl.append(1.0 if ok else 0.0)
    return cs, fl

def score(preds, d):
    """mAP@0.5, mAP@0.5:0.95 and the raw arrays for precision / recall / F1 at any confidence (ground truth = labels of folder d)"""
    d = Path(d); gts = {k: rd(d / "labels" / (Path(k).stem + ".txt")) for k in preds}; n_gt = sum(len(g) for g in gts.values()); aps = []; conf50 = tp50 = None
    for t in np.arange(0.5, 0.96, 0.05):
        cs, fl = [], []
        for k, pr in preds.items():
            c, f = match_flags(gts[k], pr, t); cs += c; fl += f
        cs, fl = np.array(cs), np.array(fl); aps.append(_ap(cs, fl, n_gt))
        if abs(t - 0.5) < 1e-9: conf50, tp50 = cs, fl
    return dict(AP50=aps[0], AP50_95=float(np.mean(aps)), conf=conf50, tp=tp50, n_gt=n_gt)

def at_conf(s, c):
    m = s["conf"] >= c; tp = s["tp"][m].sum(); p = tp / max(1, m.sum()); r = tp / max(1, s["n_gt"]); return p, r, 2 * p * r / max(1e-9, p + r)

def best_conf(s):
    cs = np.arange(0.05, 0.9, 0.025); return float(max(cs, key=lambda c: at_conf(s, c)[2]))

def rows_at(preds, d, thr):
    """per photo: true positives, false positives, false negatives, zones of your boxes, zones of the model boxes, n your boxes, n model boxes"""
    rows = []
    for k, pr in preds.items():
        gt = rd(Path(d) / "labels" / (Path(k).stem + ".txt")); pc = [b for b, c in pr if c >= thr]; _, fl = match_flags(gt, [(b, 1.0) for b in pc], 0.5); tp = int(sum(fl))
        cell = lambda b: min(2, int(b[1] * 3)) * 3 + min(2, int(b[0] * 3))
        rows.append((tp, len(pc) - tp, len(gt) - tp, {cell(g) for g in gt}, {cell(b) for b in pc}, len(gt), len(pc)))
    return rows

def prf(rows):
    tp, fp, fn = sum(r[0] for r in rows), sum(r[1] for r in rows), sum(r[2] for r in rows); p, r = tp / max(1, tp + fp), tp / max(1, tp + fn); return p, r, 2 * p * r / max(1e-9, p + r)

def zone_f1(rows):
    tp = sum(len(r[3] & r[4]) for r in rows); fp = sum(len(r[4] - r[3]) for r in rows); fn = sum(len(r[3] - r[4]) for r in rows); return 2 * tp / max(1, 2 * tp + fp + fn)

def f1_by_iou(preds, d, thr, ious=(0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8)):
    out = {}
    for t in ious:
        tp = fp = fn = 0
        for k, pr in preds.items():
            gt = rd(Path(d) / "labels" / (Path(k).stem + ".txt")); pc = [(b, 1.0) for b, c in pr if c >= thr]; _, fl = match_flags(gt, pc, t); x = int(sum(fl)); tp += x; fp += len(pc) - x; fn += len(gt) - x
        p, r = tp / max(1, tp + fp), tp / max(1, tp + fn); out[t] = 2 * p * r / max(1e-9, p + r)
    return out

def bootstrap_f1(rows, n=1000, seed=0):
    rng = np.random.default_rng(seed); idx = np.arange(len(rows)); b = [prf([rows[i] for i in rng.choice(idx, len(idx))])[2] for _ in range(n)]; return float(np.percentile(b, 2.5)), float(np.percentile(b, 97.5))

def show_examples(preds, d, thr, names, cols=4, title=""):
    d = Path(d); fig, ax = plt.subplots(-(-len(names) // cols), cols, figsize=(4 * cols, 4 * -(-len(names) // cols))); ax = np.array(ax).reshape(-1)
    for a in ax: a.axis("off")
    for a, n in zip(ax, names):
        p = next(iter((d / "images").glob(Path(n).stem + ".*"))); im = Image.open(p).convert("RGB"); W, H = im.size; dr = ImageDraw.Draw(im); lw = max(2, W // 250)
        for b in rd(d / "labels" / (Path(n).stem + ".txt")): x0, y0, x1, y1 = xyxy(b); dr.rectangle([x0 * W, y0 * H, x1 * W, y1 * H], outline=(40, 220, 60), width=lw)
        for b, c in preds[n]:
            if c >= thr: x0, y0, x1, y1 = xyxy(b); dr.rectangle([x0 * W, y0 * H, x1 * W, y1 * H], outline=(255, 40, 40), width=lw)
        a.imshow(im); a.set_title(n[:28], fontsize=8)
    fig.suptitle(title + "  (green = your boxes, red = model boxes above the threshold)"); plt.tight_layout(); plt.show()

def curves(run_dirs, labels):
    fig, ax = plt.subplots(1, 3, figsize=(15, 4))
    for rd_, lb in zip(run_dirs, labels):
        f = Path(rd_) / "results.csv"
        if not f.exists(): continue
        r = pd.read_csv(f); r.columns = [c.strip() for c in r.columns]
        ax[0].plot(r["epoch"], r["train/box_loss"], label=lb + " train"); ax[0].plot(r["epoch"], r["val/box_loss"], "--", label=lb + " val")
        ax[1].plot(r["epoch"], r["metrics/mAP50(B)"], label=lb); ax[2].plot(r["epoch"], r["metrics/mAP50-95(B)"], label=lb)
    for a, t in zip(ax, ("box loss (training = solid, validation = dashed)", "validation mAP@0.5", "validation mAP@0.5:0.95")): a.set_title(t, fontsize=10); a.set_xlabel("epoch"); a.legend(fontsize=7); a.grid(alpha=.3)
    plt.tight_layout(); plt.show()

## 2. Build the fresh test set from the photos you saved
Only photos you saved are used. A saved photo with boxes is a positive; a photo you marked **X** (no acne) is a clear-skin negative.

In [ ]:
meta = FRESH_FOLDER / "_drafts" / "meta.json"; ref = json.load(open(meta))["drafted_at"]
S = next(p for p in sorted(FRESH_FOLDER.iterdir()) if p.is_dir() and (p / "labels").is_dir() and (p / "images").is_dir() and not p.name.lower().startswith("boxed"))      # the working folder, NOT the export folder "boxed_..." (that one holds only the photos with boxes)
shutil.rmtree(FRESH_DIR, ignore_errors=True); (FRESH_DIR / "images").mkdir(parents=True); (FRESH_DIR / "labels").mkdir(parents=True); npos = nneg = 0
for f in sorted((S / "labels").glob("*.txt")):
    if f.stat().st_mtime <= ref + 1: continue
    img = next(iter((S / "images").glob(f.stem + ".*"))); shutil.copy2(img, FRESH_DIR / "images" / img.name); shutil.copy2(f, FRESH_DIR / "labels" / f.name)
    if rd(f): npos += 1
    else: nneg += 1
print("fresh test set: %d positives (with boxes), %d negatives (X, clear skin), %d photos in total" % (npos, nneg, npos + nneg)); assert npos + nneg > 0
if not SMOKE and npos + nneg < 90: print("WARNING: only %d of about 100 photos are saved. Finish boxing before scoring (the score is taken only once)." % (npos + nneg))

## 3. Score once (both models, same photos)

In [ ]:
PRED, FUSED = {}, {}
for name, m in MODELS.items():
    PRED[name] = [raw_preds(w, FRESH_DIR, m["imgsz"]) for w in m["weights"]]; FUSED[name] = {k: fuse([p[k] for p in PRED[name]]) for k in PRED[name][0]}
neg = [k for k in FUSED["installed_800"] if not rd(FRESH_DIR / "labels" / (Path(k).stem + ".txt"))]
if LOCK.exists() and not ALLOW_RESCORE:
    R = json.loads(LOCK.read_text()); print("the fresh test was already scored on", R["scored_at"], "- showing the stored result")
else:
    R = dict(scored_at=time.strftime("%Y-%m-%d %H:%M"), n_photos=len(FUSED["installed_800"]), n_negatives=len(neg), models={})
    for name, m in MODELS.items():
        st = score(FUSED[name], FRESH_DIR); rows_f = rows_at(FUSED[name], FRESH_DIR, m["thr"]); p, r, f = prf(rows_f); lo, hi = bootstrap_f1(rows_f)
        fa = float(np.mean([any(c >= m["thr"] for b, c in FUSED[name][k]) for k in neg])) if neg else float("nan")
        R["models"][name] = dict(label=m["label"], threshold=m["thr"], mAP50=st["AP50"], mAP50_95=st["AP50_95"], precision=p, recall=r, F1=f, F1_ci=[lo, hi], zone_F1=zone_f1(rows_f), false_alarm_rate=fa,
                                 single_seed_mAP50=[score(a, FRESH_DIR)["AP50"] for a in PRED[name]])
    LOCK.parent.mkdir(parents=True, exist_ok=True); LOCK.write_text(json.dumps(R, indent=1)); print("scored and stored in", LOCK)
print("FRESH TEST: %d photos, of which %d clear-skin negatives" % (R["n_photos"], R["n_negatives"]))
tbl = pd.DataFrame([dict(model=v["label"], threshold=v["threshold"], mAP50=v["mAP50"], mAP50_95=v["mAP50_95"], precision=v["precision"], recall=v["recall"], F1=v["F1"], F1_range="%.2f-%.2f" % tuple(v["F1_ci"]),
                         zone_F1=v["zone_F1"], false_alarm_on_clear_skin=v["false_alarm_rate"]) for v in R["models"].values()])
display(tbl.round(3))
for k, v in R["models"].items(): print("%-14s single seeds, mAP@0.5: %s" % (k, [round(x, 3) for x in v["single_seed_mAP50"]]))

## 4. The decision (rule fixed in advance, see the top of the notebook)

In [ ]:
c, i = R["models"]["candidate_960"], R["models"]["installed_800"]
d_map = c["mAP50"] - i["mAP50"]; d_p = c["precision"] - i["precision"]; d_r = c["recall"] - i["recall"]; d_fa = c["false_alarm_rate"] - i["false_alarm_rate"]
checks = [("mAP@0.5 at least +0.02 above the installed model", d_map >= 0.02, "%+.3f" % d_map), ("precision not more than 0.02 lower", d_p >= -0.02, "%+.3f" % d_p),
          ("recall not more than 0.02 lower", d_r >= -0.02, "%+.3f" % d_r), ("false alarms on clear skin not more than 5 points higher", (d_fa != d_fa) or d_fa <= 0.05, "n/a (no clear-skin photos in this set)" if d_fa != d_fa else "%+.1f points" % (100 * d_fa))]
for name, ok, val in checks: print("%-62s %-5s %s" % (name, "PASS" if ok else "FAIL", val))
print("\nVERDICT:", "the candidate passes every check: it may be installed (after you and Claude confirm)" if all(ok for _, ok, _ in checks) else "the candidate does NOT pass every check: the installed model stays")
print("Reaches 0.50 mAP@0.5 on the fresh test?  candidate: %s (%.3f) | installed: %s (%.3f)" % ("YES" if c["mAP50"] >= 0.5 else "no", c["mAP50"], "YES" if i["mAP50"] >= 0.5 else "no", i["mAP50"]))

## 5. Precision-recall curves and true / false / missed boxes

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8)); cols_ = {"candidate_960": "#e45756", "installed_800": "#4c78a8"}; cs = np.arange(0.05, 0.9, 0.0125); counts = {}
for name, m in MODELS.items():
    sc = score(FUSED[name], FRESH_DIR); pr_ = [at_conf(sc, c_) for c_ in cs]; ax[0].plot([x[1] for x in pr_], [x[0] for x in pr_], color=cols_[name], label="%s (mAP@0.5 %.3f)" % (m["label"], sc["AP50"]))
    rr = rows_at(FUSED[name], FRESH_DIR, m["thr"]); counts[name] = (sum(x[0] for x in rr), sum(x[1] for x in rr), sum(x[2] for x in rr))
ax[0].set_xlabel("recall"); ax[0].set_ylabel("precision"); ax[0].set_title("precision-recall curve (IoU 0.5)"); ax[0].legend(fontsize=8); ax[0].grid(alpha=.3); ax[0].set_xlim(0, 1); ax[0].set_ylim(0, 1)
x = np.arange(3); wd = 0.38
for j, (name, m) in enumerate(MODELS.items()):
    ax[1].bar(x + (j - 0.5) * wd, counts[name], wd, color=cols_[name], label=m["label"])
    for xi, v in zip(x + (j - 0.5) * wd, counts[name]): ax[1].text(xi, v, str(v), ha="center", va="bottom", fontsize=8)
ax[1].set_xticks(x); ax[1].set_xticklabels(["found (TP)", "false boxes (FP)", "missed (FN)"]); ax[1].legend(fontsize=8); ax[1].set_title("at each model's threshold")
plt.tight_layout(); plt.show()

## 6. Where it works: by photo resolution, by spot density, and on clear skin (both models)

In [ ]:
from PIL import Image
info = {}
for k in FUSED["installed_800"]:
    p = next(iter((FRESH_DIR / "images").glob(Path(k).stem + ".*"))); w_, h_ = Image.open(p).size; info[k] = min(w_, h_)
def pos(k): return rd(FRESH_DIR / "labels" / (Path(k).stem + ".txt"))
out = []
for name, m in MODELS.items():
    for gname, fn_ in (("low resolution (<500 px)", lambda k: info[k] < 500), ("high resolution (>=500 px)", lambda k: info[k] >= 500), ("sparse (1-6 spots)", lambda k: 1 <= len(pos(k)) <= 6),
                       ("medium (7-14)", lambda k: 7 <= len(pos(k)) <= 14), ("dense (15+)", lambda k: len(pos(k)) >= 15)):
        ks = [k for k in FUSED[name] if pos(k) and fn_(k)]
        if not ks: continue
        p, r, f = prf(rows_at({k: FUSED[name][k] for k in ks}, FRESH_DIR, m["thr"])); out.append(dict(model=m["label"], group=gname, photos=len(ks), precision=p, recall=r, F1=f))
display(pd.DataFrame(out).round(3))
if neg:
    tt = []
    for name, m in MODELS.items():
        for t in (0.2, 0.25, 0.275, 0.3, 0.35, 0.4):
            n = np.array([sum(1 for b, c in FUSED[name][k] if c >= t) for k in neg]); tt.append(dict(model=m["label"], threshold=t, photos_with_a_false_box=int((n > 0).sum()), of=len(n), specificity=1 - (n > 0).mean(), false_boxes_per_photo=n.mean()))
    display(pd.DataFrame(tt).round(3))

## 7. Example predictions of the candidate (green = your boxes, red = model boxes above the threshold)

In [ ]:
names = [p.name for p in sorted((FRESH_DIR / "images").glob("*"))]
posn = [n for n in names if pos(n)][:6]; negn = [n for n in names if not pos(n)][:2]
show_examples(FUSED["candidate_960"], FRESH_DIR, MODELS["candidate_960"]["thr"], posn + negn, cols=4, title="fresh test photos, 960 px candidate")

## 8. Summary for Chapter 4
Copy the numbers from section 3 and the verdict from section 4. State: how many fresh photos (positives and negatives), that they were never used for training, validation, tuning or threshold choice, that you boxed them from scratch without model drafts, that the result was computed once, the 95 % range, and the false-alarm rate. Report the result whether or not it reaches 0.50.